# AEON v3.0 — Colab launcher

This single cell pulls the latest `aeon.py` from GitHub, installs dependencies, and runs it.

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Secrets (left sidebar) → add `HUGGINGFACE_TOKEN`, `GITHUB_TOKEN` (optional), `SUPABASE_URL` + `SUPABASE_ANON_KEY` (optional). Tick **Notebook access** for each.
3. Click the play button ▶︎


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  AEON v3.0 — Colab launcher (single cell)                                  ║
# ║  1. Clone/pull https://github.com/beatznlg/aeon.git into /content/aeon       ║
# ║  2. pip install -r requirements.txt                                        ║
# ║  3. exec(open("aeon.py").read())                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, subprocess, sys, pathlib, json

REPO = "https://github.com/beatznlg/aeon.git"
WORKDIR = pathlib.Path("/content/aeon")

# Secrets are injected by Colab; also allow plain os.environ for local testing.
def _get(k):
    try:
        from google.colab import userdata
        v = userdata.get(k)
        if v:
            os.environ[k] = v
            return v
    except Exception:
        pass
    return os.getenv(k)

for _k in ["HUGGINGFACE_TOKEN", "GITHUB_TOKEN", "SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY"]:
    _get(_k)

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")

if WORKDIR.exists():
    print("[launcher] pulling latest in", WORKDIR)
    subprocess.check_call(["git", "pull", "--rebase", "--autostash"], cwd=str(WORKDIR))
else:
    print("[launcher] cloning", REPO, "->", WORKDIR)
    clone_cmd = ["git", "clone", REPO, str(WORKDIR)]
    if GITHUB_TOKEN:
        auth_repo = REPO.replace("https://", f"https://{GITHUB_TOKEN}@")
        clone_cmd = ["git", "clone", auth_repo, str(WORKDIR)]
    subprocess.check_call(clone_cmd)

os.chdir(str(WORKDIR))

# Show which commit we are on.
_head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"[launcher] cwd = {WORKDIR} | HEAD = {_head}")

# Install dependencies incrementally.
print("[launcher] installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# Run the kernel.
print("[launcher] executing aeon.py...")
exec(open("aeon.py").read())
